# Capstone Notebook — Refresh / Content Opportunity Scoring

End-to-end: synthetic FlyRank-schema panel → time-aware leakage-checked features/labels → Random Forest vs two baselines, validated two ways → charts for the paper → a ranked action engine with reason codes for every currently-live page.

This notebook stitches together assignments 01–03 and adds the final deliverable: the ranked action engine that the paper's recommendations section is built from.

## 1. Load panel (see 01_eda.ipynb for exploration)

In [ ]:
import pandas as pd
df = pd.read_parquet('../flyrank_synthetic_weekly.parquet')
df.shape

## 2. Features, labels, leakage checks (full logic in 02_feature_engineering.ipynb)

In [ ]:
# See 02_feature_engineering.ipynb for the full, annotated version of this step.
# Re-run here for a self-contained capstone notebook:
import numpy as np, pandas as pd

df = pd.read_parquet("/home/claude/capstone/flyrank_synthetic_weekly.parquet")

def expected_ctr(p):
    return 0.32 / (1 + p)**0.9

FEATURE_WINDOW = 12   # weeks of history used for features
LABEL_WINDOW = 6      # weeks ahead used to define the label
CUTOFFS_TRAIN = [14, 18, 22, 26, 30, 34]   # all history+label fully resolves by week 40
CUTOFFS_TEST  = [52, 56, 60, 64, 68, 72]   # all history starts week>=41, fully after the week-40 gap
GAP_CHECK_END_TRAIN = max(CUTOFFS_TRAIN) + LABEL_WINDOW          # 32
GAP_CHECK_START_TEST = min(CUTOFFS_TEST) - FEATURE_WINDOW + 1    # 23

def build_sample(g, cutoff):
    """g: single page's full 52-week df, sorted by week. cutoff = last week INCLUDED in feature window."""
    hist = g[(g.week > cutoff - FEATURE_WINDOW) & (g.week <= cutoff)]
    fut  = g[(g.week > cutoff) & (g.week <= cutoff + LABEL_WINDOW)]
    if len(hist) < FEATURE_WINDOW or len(fut) < LABEL_WINDOW:
        return None

    last4 = hist[hist.week > cutoff - 4]
    prior4 = hist[(hist.week > cutoff - 8) & (hist.week <= cutoff - 4)]
    first_half = hist[hist.week <= cutoff - 6]
    second_half = hist[hist.week > cutoff - 6]

    recent_avg_clicks = last4.clicks.mean()
    prior_avg_clicks = prior4.clicks.mean() if len(prior4) else np.nan
    momentum_clicks = (recent_avg_clicks - prior_avg_clicks) / max(prior_avg_clicks, 1e-6)

    recent_avg_pos = last4.avg_position.mean()
    prior_avg_pos = prior4.avg_position.mean() if len(prior4) else np.nan
    momentum_position = (prior_avg_pos - recent_avg_pos)  # positive = improved (moved up)

    early_slope = second_half.clicks.mean() - first_half.clicks.mean()  # past trend within window
    volatility_clicks = hist.clicks.std() / max(hist.clicks.mean(), 1e-6)

    exp_ctr = expected_ctr(recent_avg_pos)
    actual_ctr = last4.ctr.mean()
    ctr_gap = actual_ctr - exp_ctr

    future_avg_clicks = fut.clicks.mean()
    future_change = (future_avg_clicks - recent_avg_clicks) / max(recent_avg_clicks, 1e-6)

    if future_change > 0.15 and early_slope < 0:
        label = "recovering"
    elif future_change > 0.15:
        label = "growing"
    elif future_change < -0.15:
        label = "declining"
    else:
        label = "stable"

    row = dict(
        page_id=g.page_id.iloc[0], cutoff_week=cutoff, content_type=g.content_type.iloc[0],
        word_count=g.word_count.iloc[0], internal_links=g.internal_links.iloc[0],
        has_schema_markup=bool(g.has_schema_markup.iloc[0]),
        publish_age_days=hist.publish_age_days.iloc[-1],
        days_since_update=hist.days_since_update.iloc[-1],
        recent_avg_clicks=recent_avg_clicks, recent_avg_impressions=last4.impressions.mean(),
        recent_avg_position=recent_avg_pos, recent_avg_ctr=actual_ctr,
        momentum_clicks_4v8=momentum_clicks, momentum_position_4v8=momentum_position,
        early_within_window_slope=early_slope, volatility_clicks=volatility_clicks,
        ctr_gap_vs_expected=ctr_gap, engagement_rate=last4.engagement_rate.mean(),
        label=label, _archetype_debug=g.archetype.iloc[0],
    )
    return row

samples = []
for pid, g in df.groupby("page_id"):
    g = g.sort_values("week")
    for c in CUTOFFS_TRAIN + CUTOFFS_TEST:
        r = build_sample(g, c)
        if r: samples.append(r)

samp = pd.DataFrame(samples)
print("Total samples:", len(samp))
print(samp.label.value_counts())

# ---- leakage / window sanity checks ----
assert GAP_CHECK_END_TRAIN <= GAP_CHECK_START_TEST - 1, "train/test windows overlap in time!"
print(f"Train windows fully resolve by week {GAP_CHECK_END_TRAIN}; "
      f"earliest test feature data starts week {GAP_CHECK_START_TEST}. Gap = "
      f"{GAP_CHECK_START_TEST - GAP_CHECK_END_TRAIN} week(s). No leakage across the split.")

# grouped page holdout: 30% of pages never appear in ANY training sample
rng = np.random.default_rng(7)
all_pages = samp.page_id.unique()
holdout_pages = set(rng.choice(all_pages, size=int(0.3*len(all_pages)), replace=False))

train_mask = samp.cutoff_week.isin(CUTOFFS_TRAIN) & (~samp.page_id.isin(holdout_pages))
test_time_mask = samp.cutoff_week.isin(CUTOFFS_TEST) & (~samp.page_id.isin(holdout_pages))          # time-holdout, same pages
test_page_time_mask = samp.cutoff_week.isin(CUTOFFS_TEST) & (samp.page_id.isin(holdout_pages))       # page+time holdout, unseen pages

train = samp[train_mask].copy()
test_time = samp[test_time_mask].copy()
test_page_time = samp[test_page_time_mask].copy()

print("\nTrain:", len(train), "| Test (time-holdout, seen pages):", len(test_time),
      "| Test (page+time holdout, unseen pages):", len(test_page_time))

overlap = set(train.page_id) & set(test_page_time.page_id)
assert len(overlap) == 0, "page leakage into strict holdout!"
print("Confirmed zero page overlap between train and the page+time holdout set.")

train.to_parquet("/home/claude/capstone/train.parquet", index=False)
test_time.to_parquet("/home/claude/capstone/test_time.parquet", index=False)
test_page_time.to_parquet("/home/claude/capstone/test_page_time.parquet", index=False)
samp.to_parquet("/home/claude/capstone/all_samples.parquet", index=False)


## 3. Model + baselines + validation (full logic in 03_modeling_validation.ipynb)

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import json

train = pd.read_parquet("/home/claude/capstone/train.parquet")
test_time = pd.read_parquet("/home/claude/capstone/test_time.parquet")
test_page_time = pd.read_parquet("/home/claude/capstone/test_page_time.parquet")

NUM_FEATS = ["word_count","internal_links","publish_age_days","days_since_update",
             "recent_avg_clicks","recent_avg_impressions","recent_avg_position","recent_avg_ctr",
             "momentum_clicks_4v8","momentum_position_4v8","early_within_window_slope",
             "volatility_clicks","ctr_gap_vs_expected","engagement_rate"]
CAT_FEATS = ["content_type","has_schema_markup"]
LABELS = ["declining","stable","recovering","growing"]

def Xy(d):
    return d[NUM_FEATS+CAT_FEATS], d["label"]

Xtr, ytr = Xy(train)
Xte1, yte1 = Xy(test_time)
Xte2, yte2 = Xy(test_page_time)

pre = ColumnTransformer([
    ("num","passthrough", NUM_FEATS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
])

model = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_leaf=3,
                                    class_weight="balanced", random_state=42))
])
model.fit(Xtr, ytr)

# ---- Baseline 1: majority class ----
maj_class = ytr.value_counts().idxmax()
def majority_pred(y): return np.array([maj_class]*len(y))

# ---- Baseline 2: naive single-signal heuristic an analyst might use by eye ----
def heuristic_pred(d):
    out = []
    for m in d["momentum_clicks_4v8"]:
        if m > 0.15: out.append("growing")
        elif m < -0.15: out.append("declining")
        else: out.append("stable")
    return np.array(out)

results = {}
for name, X, y in [("test_time (same pages, future window)", Xte1, yte1),
                    ("test_page_time (unseen pages, future window)", Xte2, yte2)]:
    pred_model = model.predict(X)
    pred_maj = majority_pred(y)
    pred_heur = heuristic_pred(X)
    results[name] = {
        "model_macro_f1": f1_score(y, pred_model, average="macro", labels=LABELS, zero_division=0),
        "model_accuracy": (pred_model==y.values).mean(),
        "majority_baseline_macro_f1": f1_score(y, pred_maj, average="macro", labels=LABELS, zero_division=0),
        "majority_baseline_accuracy": (pred_maj==y.values).mean(),
        "heuristic_baseline_macro_f1": f1_score(y, pred_heur, average="macro", labels=LABELS, zero_division=0),
        "heuristic_baseline_accuracy": (pred_heur==y.values).mean(),
        "model_report": classification_report(y, pred_model, labels=LABELS, zero_division=0, output_dict=True),
        "confusion_matrix": confusion_matrix(y, pred_model, labels=LABELS).tolist(),
    }

print(json.dumps({k: {kk:vv for kk,vv in v.items() if kk not in ("model_report","confusion_matrix")}
                   for k,v in results.items()}, indent=2))

with open("/home/claude/capstone/results.json","w") as f:
    json.dump(results, f, indent=2)

# feature importances
importances = model.named_steps["clf"].feature_importances_
feat_names = NUM_FEATS + list(model.named_steps["pre"].named_transformers_["cat"].get_feature_names_out(CAT_FEATS))
imp_df = pd.DataFrame({"feature": feat_names, "importance": importances}).sort_values("importance", ascending=False)
imp_df.to_csv("/home/claude/capstone/feature_importance.csv", index=False)
print("\nTop features:\n", imp_df.head(10).to_string(index=False))

import joblib
joblib.dump(model, "/home/claude/capstone/refresh_opportunity_model.joblib")


## 4. Charts for the paper

In [ ]:
import json, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans", "axes.edgecolor":"#333", "axes.labelcolor":"#222",
    "text.color":"#222", "xtick.color":"#444", "ytick.color":"#444",
    "figure.facecolor":"white", "axes.facecolor":"white"
})

PALETTE = {"declining":"#C15B4A","stable":"#8C8577","recovering":"#4A8C7A","growing":"#2E6B5E"}
LABELS = ["declining","stable","recovering","growing"]

results = json.load(open("results.json"))

# 1. Model vs baselines, macro-F1, both validation regimes
fig, ax = plt.subplots(figsize=(7,4.2))
regimes = list(results.keys())
short = ["Time-holdout\n(same pages)", "Page+time holdout\n(unseen pages)"]
x = np.arange(len(regimes)); w = 0.25
model_f1 = [results[r]["model_macro_f1"] for r in regimes]
maj_f1 = [results[r]["majority_baseline_macro_f1"] for r in regimes]
heur_f1 = [results[r]["heuristic_baseline_macro_f1"] for r in regimes]
ax.bar(x-w, maj_f1, w, label="Majority-class baseline", color="#D9D3C7")
ax.bar(x, heur_f1, w, label="Single-signal heuristic baseline", color="#B7AE9C")
ax.bar(x+w, model_f1, w, label="Model (Random Forest)", color="#2E6B5E")
ax.set_xticks(x); ax.set_xticklabels(short)
ax.set_ylabel("Macro-F1 (4 classes)")
ax.set_title("Model vs. baselines — macro-F1 across two validation regimes")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("/home/claude/capstone/paper_assets/fig1_model_vs_baseline.png", dpi=160); plt.close()

# 2. Confusion matrix (time-holdout)
cm = np.array(results["test_time (same pages, future window)"]["confusion_matrix"])
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.4,4.6))
im = ax.imshow(cm_norm, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(LABELS, rotation=30, ha="right")
ax.set_yticks(range(4)); ax.set_yticklabels(LABELS)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix — time-holdout set (row-normalized)")
for i in range(4):
    for j in range(4):
        ax.text(j,i,f"{cm_norm[i,j]:.2f}", ha="center", va="center",
                 color="white" if cm_norm[i,j]>0.5 else "#222", fontsize=9)
plt.tight_layout(); plt.savefig("/home/claude/capstone/paper_assets/fig2_confusion_matrix.png", dpi=160); plt.close()

# 3. Feature importance
imp = pd.read_csv("feature_importance.csv").head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7,4.6))
ax.barh(imp.feature, imp.importance, color="#4A8C7A")
ax.set_xlabel("Random Forest importance")
ax.set_title("Which signals drive the model's predictions")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("/home/claude/capstone/paper_assets/fig3_feature_importance.png", dpi=160); plt.close()

# 4. Label distribution
samp = pd.read_parquet("all_samples.parquet")
counts = samp.label.value_counts().reindex(LABELS)
fig, ax = plt.subplots(figsize=(6.4,4))
ax.bar(counts.index, counts.values, color=[PALETTE[l] for l in counts.index])
ax.set_ylabel("Page-window samples")
ax.set_title("Class balance across all 7,200 page-window samples")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("/home/claude/capstone/paper_assets/fig4_class_balance.png", dpi=160); plt.close()

print("charts written")


## 5. Ranked action engine
Scores every page as of the most recent 12-week window, attaches predicted status, model confidence, plain-language reason codes, and a recommended action. Sorted so declining pages needing attention surface first.

In [ ]:
import pandas as pd, numpy as np, joblib

model = joblib.load("refresh_opportunity_model.joblib")
df = pd.read_parquet("flyrank_synthetic_weekly.parquet")

NUM_FEATS = ["word_count","internal_links","publish_age_days","days_since_update",
             "recent_avg_clicks","recent_avg_impressions","recent_avg_position","recent_avg_ctr",
             "momentum_clicks_4v8","momentum_position_4v8","early_within_window_slope",
             "volatility_clicks","ctr_gap_vs_expected","engagement_rate"]
CAT_FEATS = ["content_type","has_schema_markup"]

def expected_ctr(p): return 0.32/(1+p)**0.9

CUTOFF = 84  # score every page as of "now" using the most recent 12 weeks, no future needed
rows = []
for pid, g in df.groupby("page_id"):
    g = g.sort_values("week")
    hist = g[(g.week > CUTOFF-12) & (g.week <= CUTOFF)]
    if len(hist) < 12: continue
    last4 = hist[hist.week > CUTOFF-4]; prior4 = hist[(hist.week>CUTOFF-8)&(hist.week<=CUTOFF-4)]
    first_half = hist[hist.week<=CUTOFF-6]; second_half = hist[hist.week>CUTOFF-6]
    recent_avg_clicks = last4.clicks.mean(); prior_avg_clicks = prior4.clicks.mean()
    momentum_clicks = (recent_avg_clicks-prior_avg_clicks)/max(prior_avg_clicks,1e-6)
    recent_avg_pos = last4.avg_position.mean(); prior_avg_pos = prior4.avg_position.mean()
    momentum_position = prior_avg_pos - recent_avg_pos
    early_slope = second_half.clicks.mean() - first_half.clicks.mean()
    volatility = hist.clicks.std()/max(hist.clicks.mean(),1e-6)
    exp_ctr = expected_ctr(recent_avg_pos); actual_ctr = last4.ctr.mean(); ctr_gap = actual_ctr-exp_ctr
    rows.append(dict(page_id=pid, content_type=g.content_type.iloc[0], word_count=g.word_count.iloc[0],
        internal_links=g.internal_links.iloc[0], has_schema_markup=bool(g.has_schema_markup.iloc[0]),
        publish_age_days=hist.publish_age_days.iloc[-1], days_since_update=hist.days_since_update.iloc[-1],
        recent_avg_clicks=recent_avg_clicks, recent_avg_impressions=last4.impressions.mean(),
        recent_avg_position=recent_avg_pos, recent_avg_ctr=actual_ctr,
        momentum_clicks_4v8=momentum_clicks, momentum_position_4v8=momentum_position,
        early_within_window_slope=early_slope, volatility_clicks=volatility,
        ctr_gap_vs_expected=ctr_gap, engagement_rate=last4.engagement_rate.mean()))

cur = pd.DataFrame(rows)
X = cur[NUM_FEATS+CAT_FEATS]
proba = model.predict_proba(X)
classes = model.named_steps["clf"].classes_
pred = model.predict(X)
cur["predicted_status"] = pred
for i,c in enumerate(classes):
    cur[f"p_{c}"] = proba[:,i]
cur["confidence"] = proba.max(axis=1)

# population medians for reason-code context
med = cur[NUM_FEATS].median()

def reason_codes(row, k=3):
    reasons = []
    if row.momentum_clicks_4v8 < -0.15:
        reasons.append(f"click momentum -{abs(row.momentum_clicks_4v8)*100:.0f}% vs prior 4wk")
    elif row.momentum_clicks_4v8 > 0.15:
        reasons.append(f"click momentum +{row.momentum_clicks_4v8*100:.0f}% vs prior 4wk")
    if row.momentum_position_4v8 > 1.5:
        reasons.append(f"avg position improved {row.momentum_position_4v8:.1f} spots")
    elif row.momentum_position_4v8 < -1.5:
        reasons.append(f"avg position slipped {abs(row.momentum_position_4v8):.1f} spots")
    if row.ctr_gap_vs_expected < -med_ctr_gap_std:
        reasons.append(f"CTR trails position-expected by {abs(row.ctr_gap_vs_expected)*100:.1f}pp — metadata review candidate")
    elif row.ctr_gap_vs_expected > med_ctr_gap_std:
        reasons.append(f"CTR beats position-expected by {row.ctr_gap_vs_expected*100:.1f}pp")
    if row.days_since_update > 365:
        reasons.append(f"not updated in {int(row.days_since_update)}d")
    if row.volatility_clicks > cur.volatility_clicks.quantile(0.8):
        reasons.append("high week-to-week volatility")
    if row.early_within_window_slope < -med_slope_std and row.momentum_clicks_4v8 > 0:
        reasons.append("was declining, now turning up")
    if not reasons:
        reasons.append("within normal range on tracked signals")
    return "; ".join(reasons[:k])

med_ctr_gap_std = cur.ctr_gap_vs_expected.std()*0.5
med_slope_std = cur.early_within_window_slope.std()*0.5
cur["reason_codes"] = cur.apply(reason_codes, axis=1)

def action_for(row):
    s = row.predicted_status
    if s == "declining":
        return "Refresh / rewrite" if row.days_since_update>300 else "Review for cannibalization or SERP change"
    if s == "recovering":
        return "Protect & monitor — reinforce what's working"
    if s == "growing":
        return "Expand — add internal links / related content"
    return "Monitor" if row.ctr_gap_vs_expected < -0.02 else "Protect"

cur["recommended_action"] = cur.apply(action_for, axis=1)

priority = {"declining":0,"recovering":1,"growing":2,"stable":3}
cur["priority_rank"] = cur.predicted_status.map(priority)
cur = cur.sort_values(["priority_rank","confidence"], ascending=[True,False]).reset_index(drop=True)
cur.insert(0, "rank", range(1, len(cur)+1))

out_cols = ["rank","page_id","content_type","predicted_status","confidence","recommended_action",
            "reason_codes","recent_avg_clicks","recent_avg_position","momentum_clicks_4v8","days_since_update"]
cur[out_cols].to_csv("/home/claude/capstone/ranked_action_engine.csv", index=False)
print(cur[out_cols].head(15).to_string(index=False))
print("\nStatus counts:\n", cur.predicted_status.value_counts())


## Output
`ranked_action_engine.csv` is the artifact the paper's Ranked Recommendations section summarizes. `results.json` and `paper_assets/*.png` back the Results section.